In [ ]:
# RSNA Knee - training-data reconnaissance (no GPU, no submission).
# Characterize the real supervision so the label pipeline + CV can be designed.
import os, glob, json
import pandas as pd
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)

roots = sorted(glob.glob('/kaggle/input/*'))
print('INPUT ROOTS:', roots)
base = None
for r in roots:
    if 'rsna' in r.lower() or 'knee' in r.lower():
        base = r; break
base = base or (roots[0] if roots else '/kaggle/input')
print('BASE:', base)
print('TOP-LEVEL:', sorted(os.listdir(base))[:80])


In [ ]:
# Every CSV under the competition dir: shape, columns, head.
csvs = sorted(glob.glob(base + '/**/*.csv', recursive=True))
print('CSV FILES:', len(csvs))
frames = {}
for c in csvs:
    sz = os.path.getsize(c)
    try:
        df = pd.read_csv(c)
        frames[os.path.basename(c)] = df
        print(f'\n=== {c}  ({sz:,} bytes)  shape={df.shape} ===')
        print('cols:', list(df.columns))
        print(df.head(4).to_string())
    except Exception as e:
        print(f'\n=== {c}  ({sz:,} bytes)  READ-ERROR: {e} ===')


In [ ]:
# Non-CSV small files (reports may be json/txt/parquet). List + peek.
others = []
for ext in ('*.json','*.txt','*.parquet','*.tsv','*.jsonl'):
    others += glob.glob(base + '/**/' + ext, recursive=True)
print('NON-CSV data files:', len(others))
for f in sorted(others)[:40]:
    print(f'  {f}  ({os.path.getsize(f):,} bytes)')
for f in sorted(others)[:3]:
    print(f'\n--- peek {f} ---')
    try:
        with open(f, encoding='utf-8', errors='replace') as fh:
            print(fh.read(1500))
    except Exception as e:
        print('peek error', e)


In [ ]:
# Label supervision: find the train-label frame, report prevalence per label,
# and how many studies are gold-labeled vs report-only vs uncertain (-1/NaN).
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
for name, df in frames.items():
    hit = [l for l in LABELS if l in df.columns]
    if len(hit) >= 6:
        print(f'\n### LABEL FRAME: {name}  shape={df.shape}  ({len(hit)}/12 label cols)')
        print('columns:', list(df.columns))
        vals = pd.unique(df[hit].values.ravel())
        print('distinct label values:', sorted([str(v) for v in vals])[:20])
        print('\nper-label value counts:')
        for l in hit:
            vc = df[l].value_counts(dropna=False).to_dict()
            print(f'  {l:20s} {vc}')
        for flag in ('is_gold','gold','split','source','label_source','expert'):
            if flag in df.columns:
                print(f'\n{flag} counts:', df[flag].value_counts(dropna=False).to_dict())


In [ ]:
# Train image layout: studies / series / sequence-metadata distribution.
for d in ('train','train_series','train_images'):
    p = os.path.join(base, d)
    if os.path.isdir(p):
        subs = os.listdir(p)
        print(f'{d}/: {len(subs)} entries; sample:', subs[:3])
ts = frames.get('train_series.csv')
if ts is not None:
    print('\ntrain_series studies:', ts['StudyInstanceUID'].nunique() if 'StudyInstanceUID' in ts else '?')
    for col in ('Anatomical_Plane','Fluid_Sensitive','Fat_Suppression'):
        if col in ts.columns:
            print(f'{col}:', ts[col].value_counts(dropna=False).to_dict())
print('\nDONE')
